## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-04-10-56-27 +0000,nypost,US moves to drop frog embryo smuggling charges...,https://nypost.com/2026/11/04/us-news/us-moves...
1,2026-11-04-10-54-18 +0000,nyt,Here’s the latest.,https://www.nytimes.com/live/2026/11/04/us/mid...
2,2026-11-04-10-48-50 +0000,nyt,What’s Next in the Alaska Senate Race?,https://www.nytimes.com/2026/11/04/us/alaska-s...
3,2026-11-04-10-48-00 +0000,wsj,Americans Cut Back on Boats and RVs—but Can’t ...,https://www.wsj.com/business/retail/americans-...
4,2026-11-04-10-47-31 +0000,bbc,What's happening to UK interest rates and what...,https://www.bbc.co.uk/news/articles/c3dky111m4...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2602/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
14,race,54
97,trump,49
94,democrats,46
268,election,45
13,senate,38


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
199,2026-11-04-04-40-50 +0000,chitri,Iowa Senate race: Democrat Josh Turek wins ele...,https://www.chicagotribune.com/2026/11/03/josh...,263
311,2026-11-03-22-49-00 +0000,wsj,Republicans and Democrats face off in a high-s...,https://www.wsj.com/politics/policy/democrats-...,260
152,2026-11-04-05-28-25 +0000,chitri,"Iowa governor race: Rob Sand wins election, fl...",https://www.chicagotribune.com/2026/11/03/rob-...,244
236,2026-11-04-03-14-56 +0000,chitri,South Carolina Senate race: Democrat Annie And...,https://www.chicagotribune.com/2026/11/03/sout...,236
70,2026-11-04-08-01-04 +0000,startribune,Democrats win the US House and Senate as voter...,https://www.startribune.com/midterm-elections-...,227


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
199,263,2026-11-04-04-40-50 +0000,chitri,Iowa Senate race: Democrat Josh Turek wins ele...,https://www.chicagotribune.com/2026/11/03/josh...
311,260,2026-11-03-22-49-00 +0000,wsj,Republicans and Democrats face off in a high-s...,https://www.wsj.com/politics/policy/democrats-...
205,105,2026-11-04-04-33-12 +0000,chitri,Lake County Sheriff John Idleburg declares vic...,https://www.chicagotribune.com/2026/11/03/lake...
195,97,2026-11-04-04-43-53 +0000,chitri,Wisconsin governor race: Democrat David Crowle...,https://www.chicagotribune.com/2026/11/03/demo...
268,95,2026-11-04-01-00-20 +0000,chitri,Voters usher in new era with state Rep. La Sha...,https://www.chicagotribune.com/2026/11/03/illi...
85,90,2026-11-04-07-29-00 +0000,wsj,A political reckoning has shattered President ...,https://www.wsj.com/politics/elections/a-polit...
335,72,2026-11-03-22-01-05 +0000,nyt,How Will the Midterms Play Out? These Races Ma...,https://www.nytimes.com/2026/11/03/us/politics...
495,59,2026-11-03-11-05-43 +0000,chitri,‘Absolutely astonishing’: Pastry fans line up ...,https://www.chicagotribune.com/2026/11/03/trit...
233,57,2026-11-04-03-22-53 +0000,wsj,Stock Market Today: Stock Futures Rebound as I...,https://www.wsj.com/livecoverage/stock-market-...
181,56,2026-11-04-05-07-00 +0000,wsj,President Trump said on Tuesday night that the...,https://www.wsj.com/politics/elections/trump-a...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
